# 04 · Forecast -> flexibility

Turns a Slice-3 day-ahead **interval** forecast for one feeder into a **risk-adjusted**
congestion-relief schedule: down-flex sized against the UPPER forecast bound, under the
binding (min of thermal / contractual) limit. Plots the schedule and reports volume +
timing. Runs fully offline.

In [ ]:
import os
os.environ["FORECAUS_OFFLINE"] = "1"   # run fully offline on the committed fixtures
import warnings; warnings.filterwarnings("ignore")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from pathlib import Path
# Resolve figures/ whether executed from the repo root or from notebooks/.
FIG = Path("notebooks/figures") if Path("notebooks").is_dir() else Path("figures")
FIG.mkdir(parents=True, exist_ok=True)
print("offline:", os.environ["FORECAUS_OFFLINE"], "| figures ->", FIG)


In [ ]:
from forecaus_grid_odeon.flex import run_flex
res = run_flex()
sched, s = res['schedule'], res['summary']
print(f"feeder {res['feeder_id']}  |  thermal {res['thermal_limit']:.1f} kW  |  "
      f"contractual {res['contractual_limit']:.1f} kW  |  binding {res['binding_limit']:.1f} kW")
breach = sched[(sched['flex_down'] > 0) | (sched['flex_up'] > 0)]
print(f"\n{len(breach)} active relief steps:")
print(breach[['forecast', 'upper', 'ss_limit', 'flex_down', 'flex_up']].round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sched.index, sched['forecast'], color='#4c78a8', label='forecast')
ax.fill_between(sched.index, sched['lower'], sched['upper'], alpha=0.2, color='#4c78a8', label='interval')
ax.plot(sched.index, sched['ss_limit'], ls='--', color='red', label='binding limit')
ax2 = ax.twinx()
ax2.bar(sched.index, sched['flex_down'], width=0.015, color='#e45756', alpha=0.6)
ax.set_ylabel('load [kW]'); ax2.set_ylabel('flex_down [kW]')
ax.set_title('Forecast vs limit -> risk-adjusted down-flex'); ax.legend(loc='upper left')
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(FIG / '04_forecast_to_flex.png', dpi=130); plt.close(fig)
print('saved', FIG / '04_forecast_to_flex.png')
print(f"down-flex: peak {s['peak_flex_down']:.1f} kW, energy {s['energy_flex_down_kwh']:.1f} kWh over {s['n_breach_down']} steps")
print(f"risk adjustment: {s['energy_flex_down_kwh']:.1f} kWh vs {res['summary_point']['energy_flex_down_kwh']:.1f} kWh point-only")